In [2]:
import json
import os
import requests
from datetime import datetime, timezone

VILLES = ["paris", "lyon", "marseille", "lille", "toulouse",
          "bordeaux", "nantes", "strasbourg", "nice", "montpellier"]

LAKEHOUSE_ROOT = "/lakehouse/default/Files"

with open(f"{LAKEHOUSE_ROOT}/secrets.json") as f:
    SECRETS = json.load(f)

OPENWEATHER_API_KEY = SECRETS["OPENWEATHER_API_KEY"]

print(f"Configuration chargée — {len(VILLES)} villes")

StatementMeta(, b5f3904f-f4a3-464c-b75b-edd3c167791b, 4, Finished, Available, Finished, False)

Configuration chargée — 10 villes


In [1]:
def extract_openweather(ville: str) -> dict | None:
    """Appelle l'API OpenWeatherMap pour une ville. Retourne None si erreur."""
    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": f"{ville},FR",
        "appid": OPENWEATHER_API_KEY,
        "units": "metric",
        "lang": "fr",
    }
    try:
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        return r.json()
    except requests.RequestException as e:
        print(f"  [KO] {ville} — erreur : {e}")
        return None

StatementMeta(, b5f3904f-f4a3-464c-b75b-edd3c167791b, 3, Finished, Available, Finished, False)

In [3]:
def save_to_bronze(data: dict, source: str, ville: str, ts: datetime) -> str:
    """Écrit le JSON brut dans Files/bronze/{source}/{aaaa}/{mm}/{jj}/{ville}_{hh}h.json"""
    dossier = (f"{LAKEHOUSE_ROOT}/bronze/{source}/"
               f"{ts.year}/{ts.month:02d}/{ts.day:02d}")
    os.makedirs(dossier, exist_ok=True)

    chemin = f"{dossier}/{ville}_{ts.hour:02d}h.json"
    with open(chemin, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    return chemin

StatementMeta(, b5f3904f-f4a3-464c-b75b-edd3c167791b, 5, Finished, Available, Finished, False)

In [4]:
timestamp = datetime.now(timezone.utc)
ok, ko = 0, 0

print("=" * 45)
print(f"INGESTION OPENWEATHER — {timestamp:%Y-%m-%d %H:%M} UTC")
print("=" * 45)

for ville in VILLES:
    print(f"Traitement : {ville}")
    data = extract_openweather(ville)
    if data is None:
        ko += 1
        continue
    chemin = save_to_bronze(data, "openweather", ville, timestamp)
    print(f"  [OK] {chemin.replace(LAKEHOUSE_ROOT, 'Files')}")
    ok += 1

print("-" * 45)
print(f"INGESTION OPENWEATHER TERMINÉE — OK : {ok} | KO : {ko}")

StatementMeta(, b5f3904f-f4a3-464c-b75b-edd3c167791b, 6, Finished, Available, Finished, False)

INGESTION OPENWEATHER — 2026-09-02 10:46 UTC
Traitement : paris
  [OK] Files/bronze/openweather/2026/09/02/paris_10h.json
Traitement : lyon
  [OK] Files/bronze/openweather/2026/09/02/lyon_10h.json
Traitement : marseille
  [OK] Files/bronze/openweather/2026/09/02/marseille_10h.json
Traitement : lille
  [OK] Files/bronze/openweather/2026/09/02/lille_10h.json
Traitement : toulouse
  [OK] Files/bronze/openweather/2026/09/02/toulouse_10h.json
Traitement : bordeaux
  [OK] Files/bronze/openweather/2026/09/02/bordeaux_10h.json
Traitement : nantes
  [OK] Files/bronze/openweather/2026/09/02/nantes_10h.json
Traitement : strasbourg
  [OK] Files/bronze/openweather/2026/09/02/strasbourg_10h.json
Traitement : nice
  [OK] Files/bronze/openweather/2026/09/02/nice_10h.json
Traitement : montpellier
  [OK] Files/bronze/openweather/2026/09/02/montpellier_10h.json
---------------------------------------------
INGESTION OPENWEATHER TERMINÉE — OK : 10 | KO : 0
